In [ ]:
import polars as pl

In [ ]:
# _file = 'qced_maf1e-3_loftee_olink_genes_EURunrelated'
_file = 'qced_maf1e-3_loftee_olink_genes'

!dx download project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/{_file}.parquet -o PATH_TO_FILE{_file}.parquet

In [ ]:
b2p = pl.scan_parquet('PATH_TO_FILE')
b2p.head().collect()

In [ ]:
(
    b2p
    .with_columns(
        pl.concat_str(
            [
                pl.col("CHROM"),
                pl.lit(":"),
                pl.col("POS").cast(pl.Utf8),
                pl.lit(":"),
                pl.col("REF"),
                pl.lit(":"),
                pl.col("ALT"),
            ]
        ).alias("ID")
    )
    .select(['ID', 'GT'])
    .sink_parquet('PATH_TO_FILE', engine='streaming')
)

## Get unique vars and MACs

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from plotnine import *

In [ ]:
var_ids = pl.scan_parquet('PATH_TO_FILE')
var_ids.head().collect()

In [ ]:
unq_vars = (
    var_ids
    .rename({c: c.lower() for c in var_ids.collect_schema().names()})
    .group_by('id')
    .agg(
        sc_ukb = pl.len(),
        ac_ukb = pl.sum("gt")
    )
    .sort('ac_ukb')
    .collect(engine='streaming')
)

unq_vars

In [ ]:
plt.hist(unq_vars['ac_ukb'], bins=50, log=True)
plt.show()

In [ ]:
max_ac = unq_vars['ac_ukb'].max()
max_ac

In [ ]:
unq_vars = (
    unq_vars
    .with_columns(
        chrom = pl.col("id").str.split(":").list.get(0),
        pos = pl.col("id").str.split(":").list.get(1),
        ref = pl.col("id").str.split(":").list.get(2),
        alt = pl.col("id").str.split(":").list.get(3),
        mac_ukb = (
            pl.when(pl.col('ac_ukb') > int(max_ac/2))
            .then(max_ac + 1 - pl.col('ac_ukb'))
            .otherwise(pl.col('ac_ukb'))
        )
    )
)

unq_vars

In [ ]:
plt.hist(unq_vars['mac_ukb'], bins=50, log=True)
plt.show()

In [ ]:
unq_vars.write_parquet('PATH_TO_FILE')

In [ ]:
!dx upload PATH_TO_FILE{_file}_variant_metadata.parquet --path project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/